<a href="https://colab.research.google.com/github/fauziardiantama/my-falcon-tuning/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Falcon-H1-Tiny 90M (Silas Edition)

This notebook automates the fine-tuning and uploading of the **Falcon-H1-Tiny 90M** model to Hugging Face.

In [ ]:
# 1. Clone the repository and install requirements
import os
repo_name = 'my-falcon-tuning'
if not os.path.exists(repo_name):
    !git clone https://github.com/fauziardiantama/{repo_name}.git

%cd {repo_name}

# Install standard training requirements
!pip install -q -r requirements.txt

In [ ]:
# 2. Start the fine-tuning process
!python train_falcon.py

### 3. Upload to Hugging Face (Clean State)
This cell will delete the existing repository (if any) and upload a fresh copy of the model files.

**Note:** Add your Hugging Face token to Colab Secrets with the name `HF_TOKEN`.

In [ ]:
from huggingface_hub import HfApi, create_repo, delete_repo
from google.colab import userdata
import os

# Configuration
HF_USERNAME = "Farid333"
REPO_NAME = "falcon-90m-silas-fine-tuned"
FULL_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
MODEL_PATH = "/content/my-falcon-tuning/falcon-90m-fine-tuned"

try:
    TOKEN = userdata.get('HF_TOKEN')
except:
    print("Error: HF_TOKEN not found in Colab Secrets.")
    TOKEN = None

if TOKEN:
    api = HfApi()

    # 1. DELETE existing repository
    print(f"Cleaning up existing repository: {FULL_REPO_ID}")
    try:
        delete_repo(repo_id=FULL_REPO_ID, token=TOKEN)
        print("Successfully deleted old repository.")
    except Exception as e:
        print(f"Skip deletion: {e} (Repo might not exist)")

    # 2. CREATE fresh repository
    print(f"Creating fresh repository: {FULL_REPO_ID}")
    create_repo(repo_id=FULL_REPO_ID, token=TOKEN, private=False)

    # 3. UPLOAD model files
    print("Uploading fresh model files...")
    api.upload_folder(
        folder_path=MODEL_PATH,
        repo_id=FULL_REPO_ID,
        repo_type="model",
        token=TOKEN
    )

    print(f"DONE! Your model is now at: https://huggingface.co/{FULL_REPO_ID}")
    print(f"Now go to https://huggingface.co/spaces/ggml-org/gguf-my-repo and paste '{FULL_REPO_ID}'")
else:
    print("Please add your HF_TOKEN to Colab Secrets and run this cell again.")